In [ ]:
import numpy as np
import os
import pandas as pd

from sklearn import metrics
from tqdm import tqdm

In [ ]:
def hamming_dist(n1, n2):
    # n1, n2 = int(n1), int(n2)
    # print("n1: ", n1)
    # print("n2: ", n2)
    x = n1 ^ n2
    setBits = 0

    while (x > 0):
        setBits += x & 1
        x >>= 1

    return setBits


def hamming_similarity_s(t1, t2):
    diff = hamming_dist(t1[0], t2[0]) + hamming_dist(t1[1], t2[1])
    return 1 - (diff / 128.0)


def compute_fss_similarity(df_input):
    scores = list()
    for idx, row in tqdm(df_input.iterrows()):
        score = hamming_similarity_s(tuple(row[['hashes0_1', 'hashes1_1']].values),
                                     tuple(row[['hashes0_2', 'hashes1_2']].values))
        # try:
        #     score = hamming_similarity_s(
        #         tuple(row[['hashes0_1', 'hashes1_1']].values),
        #         tuple(row[['hashes0_2', 'hashes1_2']].values)
        #     )
        # except Exception as e:
        #     print("Error occurred while computing hamming similarity:", e)
        #     print("Row content:")
        #     print(row)
        #     raise
        
        scores.append(score)
    return scores

In [ ]:
def compute_fuzzy_similarity(df_pairs, df_fss):
    df_pairs = df_pairs.merge(df_fss,
                              how='left',
                              left_on=['idb_path_1', 'fva_1'],
                              right_on=['path', 'address'])
    df_pairs.rename(columns={'hashes0': 'hashes0_1',
                             'hashes1': 'hashes1_1'
                             }, inplace=True)
    df_pairs.rename(columns={'time': 'fss_time_1'}, inplace=True)

    df_pairs = df_pairs.merge(df_fss,
                              how='left',
                              left_on=['idb_path_2', 'fva_2'],
                              right_on=['path', 'address'])
    df_pairs.rename(columns={'hashes0': 'hashes0_2',
                             'hashes1': 'hashes1_2'
                             }, inplace=True)
    df_pairs.rename(columns={'time': 'fss_time_2'}, inplace=True)

    df_pairs['sim'] = compute_fss_similarity(df_pairs)
    df_pairs = df_pairs[['idb_path_1','fva_1','idb_path_2','fva_2','sim']]
    return df_pairs

### Process Dataset-1 results

In [ ]:
DB1_PATH = "../../DBs/Dataset-1/pairs/testing/"
FSS_PATH = "../data/raw_results/FunctionSimSearch/Dataset-1"

for csv_name in [x for x in os.listdir(FSS_PATH) if x.endswith(".csv")]:
    csv_path = os.path.join(FSS_PATH, csv_name)
    print("[D] Processing {}".format(csv_name))

    df_pos = pd.read_csv(os.path.join(DB1_PATH, "pos_testing_Dataset-1.csv"), index_col=0)
    df_neg = pd.read_csv(os.path.join(DB1_PATH, "neg_testing_Dataset-1.csv"), index_col=0)
    df_pos_rank = pd.read_csv(os.path.join(DB1_PATH, "pos_rank_testing_Dataset-1.csv"), index_col=0)
    df_neg_rank = pd.read_csv(os.path.join(DB1_PATH, "neg_rank_testing_Dataset-1.csv"), index_col=0)

    df_fss = pd.read_csv(csv_path)
    df_fss.drop(df_fss[df_fss['branching_nodes'] == 'branching_nodes'].index, inplace=True)
    df_fss.reset_index(inplace=True, drop=True)
    df_fss = df_fss.astype({'hashes0': np.uint64, 'hashes1': np.uint64})

    df_pos = compute_fuzzy_similarity(df_pos, df_fss)
    df_neg = compute_fuzzy_similarity(df_neg, df_fss)
    df_pos_rank = compute_fuzzy_similarity(df_pos_rank, df_fss)
    df_neg_rank = compute_fuzzy_similarity(df_neg_rank, df_fss)

    df_pos.to_csv("../data/Dataset-1/pos_testing_Dataset-1_{}".format(csv_name), index=False)
    df_neg.to_csv("../data/Dataset-1/neg_testing_Dataset-1_{}".format(csv_name), index=False)
    df_pos_rank.to_csv("../data/Dataset-1/pos_rank_testing_Dataset-1_{}".format(csv_name), index=False)
    df_neg_rank.to_csv("../data/Dataset-1/neg_rank_testing_Dataset-1_{}".format(csv_name), index=False)

### Process Dataset-2 results

In [ ]:
DB2_PATH = "../../DBs/Dataset-2/pairs/testing/"
FSS_PATH = "../data/raw_results/FunctionSimSearch/Dataset-2"

for csv_name in [x for x in os.listdir(FSS_PATH) if x.endswith(".csv")]:
    csv_path = os.path.join(FSS_PATH, csv_name)
    print("[D] Processing {}".format(csv_name))

    df_pos = pd.read_csv(os.path.join(DB2_PATH, "pos_testing_Dataset-2.csv"), index_col=0)
    df_neg = pd.read_csv(os.path.join(DB2_PATH, "neg_testing_Dataset-2.csv"), index_col=0)
    df_pos_rank = pd.read_csv(os.path.join(DB2_PATH, "pos_rank_testing_Dataset-2.csv"), index_col=0)
    df_neg_rank = pd.read_csv(os.path.join(DB2_PATH, "neg_rank_testing_Dataset-2.csv"), index_col=0)

    df_fss = pd.read_csv(csv_path)
    df_fss.drop(df_fss[df_fss['branching_nodes'] == 'branching_nodes'].index, inplace=True)
    df_fss.reset_index(inplace=True, drop=True)
    df_fss = df_fss.astype({'hashes0': np.uint64, 'hashes1': np.uint64})

    df_pos = compute_fuzzy_similarity(df_pos, df_fss)
    df_neg = compute_fuzzy_similarity(df_neg, df_fss)
    df_pos_rank = compute_fuzzy_similarity(df_pos_rank, df_fss)
    df_neg_rank = compute_fuzzy_similarity(df_neg_rank, df_fss)

    df_pos.to_csv("../data/Dataset-2/pos_testing_Dataset-2_{}".format(csv_name), index=False)
    df_neg.to_csv("../data/Dataset-2/neg_testing_Dataset-2_{}".format(csv_name), index=False)
    df_pos_rank.to_csv("../data/Dataset-2/pos_rank_testing_Dataset-2_{}".format(csv_name), index=False)
    df_neg_rank.to_csv("../data/Dataset-2/neg_rank_testing_Dataset-2_{}".format(csv_name), index=False)

### Process Dataset-3 results

In [ ]:
DB2_PATH = "../../DBs/Dataset-3/pairs/testing/"
FSS_PATH = "../data/raw_results/FunctionSimSearch/Dataset-3"

for csv_name in [x for x in os.listdir(FSS_PATH) if x.endswith(".csv")]:
    csv_path = os.path.join(FSS_PATH, csv_name)
    print("[D] Processing {}".format(csv_name))

    df_pos = pd.read_csv(os.path.join(DB2_PATH, "pos_testing_Dataset-3.csv"), index_col=0)
    df_neg = pd.read_csv(os.path.join(DB2_PATH, "neg_testing_Dataset-3.csv"), index_col=0)
    df_pos_rank = pd.read_csv(os.path.join(DB2_PATH, "pos_rank_testing_Dataset-3.csv"), index_col=0)
    df_neg_rank = pd.read_csv(os.path.join(DB2_PATH, "neg_rank_testing_Dataset-3.csv"), index_col=0)

    df_fss = pd.read_csv(csv_path)
    df_fss.drop(df_fss[df_fss['branching_nodes'] == 'branching_nodes'].index, inplace=True)
    df_fss.reset_index(inplace=True, drop=True)
    df_fss = df_fss.astype({'hashes0': np.uint64, 'hashes1': np.uint64})

    df_pos = compute_fuzzy_similarity(df_pos, df_fss)
    df_neg = compute_fuzzy_similarity(df_neg, df_fss)
    df_pos_rank = compute_fuzzy_similarity(df_pos_rank, df_fss)
    df_neg_rank = compute_fuzzy_similarity(df_neg_rank, df_fss)

    df_pos.to_csv("../data/Dataset-3/pos_testing_Dataset-3_{}".format(csv_name), index=False)
    df_neg.to_csv("../data/Dataset-3/neg_testing_Dataset-3_{}".format(csv_name), index=False)
    df_pos_rank.to_csv("../data/Dataset-3/pos_rank_testing_Dataset-3_{}".format(csv_name), index=False)
    df_neg_rank.to_csv("../data/Dataset-3/neg_rank_testing_Dataset-3_{}".format(csv_name), index=False)

### Process Dataset-4 results

In [ ]:
DB2_PATH = "../../DBs/Dataset-4/pairs/testing/"
FSS_PATH = "../data/raw_results/FunctionSimSearch/Dataset-4"

for csv_name in [x for x in os.listdir(FSS_PATH) if x.endswith(".csv")]:
    csv_path = os.path.join(FSS_PATH, csv_name)
    print("[D] Processing {}".format(csv_name))

    df_pos = pd.read_csv(os.path.join(DB2_PATH, "pos_testing_Dataset-4.csv"), index_col=0)
    df_neg = pd.read_csv(os.path.join(DB2_PATH, "neg_testing_Dataset-4.csv"), index_col=0)
    df_pos_rank = pd.read_csv(os.path.join(DB2_PATH, "pos_rank_testing_Dataset-4.csv"), index_col=0)
    df_neg_rank = pd.read_csv(os.path.join(DB2_PATH, "neg_rank_testing_Dataset-4.csv"), index_col=0)

    df_fss = pd.read_csv(csv_path)
    df_fss.drop(df_fss[df_fss['branching_nodes'] == 'branching_nodes'].index, inplace=True)
    df_fss.reset_index(inplace=True, drop=True)
    df_fss = df_fss.astype({'hashes0': np.uint64, 'hashes1': np.uint64})

    df_pos = compute_fuzzy_similarity(df_pos, df_fss)
    df_neg = compute_fuzzy_similarity(df_neg, df_fss)
    df_pos_rank = compute_fuzzy_similarity(df_pos_rank, df_fss)
    df_neg_rank = compute_fuzzy_similarity(df_neg_rank, df_fss)

    df_pos.to_csv("../data/Dataset-4/pos_testing_Dataset-4_{}".format(csv_name), index=False)
    df_neg.to_csv("../data/Dataset-4/neg_testing_Dataset-4_{}".format(csv_name), index=False)
    df_pos_rank.to_csv("../data/Dataset-4/pos_rank_testing_Dataset-4_{}".format(csv_name), index=False)
    df_neg_rank.to_csv("../data/Dataset-4/neg_rank_testing_Dataset-4_{}".format(csv_name), index=False)

### Process Dataset-5 results

In [ ]:
DB2_PATH = "../../DBs/Dataset-5/pairs/testing/"
FSS_PATH = "../data/raw_results/FunctionSimSearch/Dataset-5"

for csv_name in [x for x in os.listdir(FSS_PATH) if x.endswith(".csv")]:
    csv_path = os.path.join(FSS_PATH, csv_name)
    print("[D] Processing {}".format(csv_name))

    df_pos = pd.read_csv(os.path.join(DB2_PATH, "pos_testing_Dataset-5.csv"), index_col=0)
    df_neg = pd.read_csv(os.path.join(DB2_PATH, "neg_testing_Dataset-5.csv"), index_col=0)
    df_pos_rank = pd.read_csv(os.path.join(DB2_PATH, "pos_rank_testing_Dataset-5.csv"), index_col=0)
    df_neg_rank = pd.read_csv(os.path.join(DB2_PATH, "neg_rank_testing_Dataset-5.csv"), index_col=0)

    df_fss = pd.read_csv(csv_path)
    df_fss.drop(df_fss[df_fss['branching_nodes'] == 'branching_nodes'].index, inplace=True)
    df_fss.reset_index(inplace=True, drop=True)
    df_fss = df_fss.astype({'hashes0': np.uint64, 'hashes1': np.uint64})

    df_pos = compute_fuzzy_similarity(df_pos, df_fss)
    df_neg = compute_fuzzy_similarity(df_neg, df_fss)
    df_pos_rank = compute_fuzzy_similarity(df_pos_rank, df_fss)
    df_neg_rank = compute_fuzzy_similarity(df_neg_rank, df_fss)

    df_pos.to_csv("../data/Dataset-5/pos_testing_Dataset-5_{}".format(csv_name), index=False)
    df_neg.to_csv("../data/Dataset-5/neg_testing_Dataset-5_{}".format(csv_name), index=False)
    df_pos_rank.to_csv("../data/Dataset-5/pos_rank_testing_Dataset-5_{}".format(csv_name), index=False)
    df_neg_rank.to_csv("../data/Dataset-5/neg_rank_testing_Dataset-5_{}".format(csv_name), index=False)

### Process Dataset-6 results

In [ ]:
DB2_PATH = "../../DBs/Dataset-6/pairs/testing/"
FSS_PATH = "../data/raw_results/FunctionSimSearch/Dataset-6"

for csv_name in [x for x in os.listdir(FSS_PATH) if x.endswith(".csv")]:
    csv_path = os.path.join(FSS_PATH, csv_name)
    print("[D] Processing {}".format(csv_name))

    df_pos = pd.read_csv(os.path.join(DB2_PATH, "pos_testing_Dataset-6.csv"), index_col=0)
    df_neg = pd.read_csv(os.path.join(DB2_PATH, "neg_testing_Dataset-6.csv"), index_col=0)
    df_pos_rank = pd.read_csv(os.path.join(DB2_PATH, "pos_rank_testing_Dataset-6.csv"), index_col=0)
    df_neg_rank = pd.read_csv(os.path.join(DB2_PATH, "neg_rank_testing_Dataset-6.csv"), index_col=0)

    df_fss = pd.read_csv(csv_path)
    df_fss.drop(df_fss[df_fss['branching_nodes'] == 'branching_nodes'].index, inplace=True)
    df_fss.reset_index(inplace=True, drop=True)
    df_fss = df_fss.astype({'hashes0': np.uint64, 'hashes1': np.uint64})

    df_pos = compute_fuzzy_similarity(df_pos, df_fss)
    df_neg = compute_fuzzy_similarity(df_neg, df_fss)
    df_pos_rank = compute_fuzzy_similarity(df_pos_rank, df_fss)
    df_neg_rank = compute_fuzzy_similarity(df_neg_rank, df_fss)

    df_pos.to_csv("../data/Dataset-6/pos_testing_Dataset-6_{}".format(csv_name), index=False)
    df_neg.to_csv("../data/Dataset-6/neg_testing_Dataset-6_{}".format(csv_name), index=False)
    df_pos_rank.to_csv("../data/Dataset-6/pos_rank_testing_Dataset-6_{}".format(csv_name), index=False)
    df_neg_rank.to_csv("../data/Dataset-6/neg_rank_testing_Dataset-6_{}".format(csv_name), index=False)